In this notebook, I am trying to apply a course I have read from the Introduction to Deeplearning course given by the MIT. What i plan to do is to generate music by using RNN (Recurrent Neural Network).
We will train a model to learn the patterns sheet music in ABC notation and the use this model to generate new music.

In [1]:
#Import pytorh and other relevant library
import torch
import torch.nn as nn
import torch.optim as optim

#Import all remaining packages
import numpy as np
import os
import time
import functools
from IPython import display as ipthondisplay
from scipy.io.wavfile import write
import pandas as pd

# Select device: use GPU if available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
	print("Warning: No GPU detected. Running on CPU.")
else:
	print(f"Using device: {torch.cuda.get_device_name(0)}")

For this project, i am going to use a dataset of music_abc_notation from hugging face which contains data from IrishMAN, KernScores, JSB Chorales

In [ ]:
!pip install datasets --quiet

from datasets import load_dataset
dataset = load_dataset("Seeker38/music_abc_notation")

#let's print one of the songs to inspect it in detail
print(dataset)
print(dataset['train'][0])

c:\Users\pavel\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Let's convert the song in ABC notation to an audio waveform and play it back.

In [ ]:
!pip install music21 midi2audio --quiet
from music21 import converter
from midi2audio import FluidSynth
from IPython.display import Audio

def abc_to_wav (abc_notation):
  #parse the abc notation into a music 21 stream
  abc_stream = converter.parse(abc_notation, format='abc')

  #save the stream as a midi file
  midi_file = "output.mid"
  abc_stream.write('midi', fp = midi_file)

  # convert the midi file to WAV using FluidSynth
  wav_file = "output.wav"
  fs = FluidSynth()
  fs.midi_to_audio(midi_file, wav_file)

  return wav_file


In [ ]:
# First, let's install FluidSynth if not already installed
!pip install pretty_midi --quiet

# Download a soundfont file if not present
import os
import urllib.request

def download_soundfont():
    soundfont_path = "default.sf2"
    if not os.path.exists(soundfont_path):
        print("Downloading soundfont file...")
        url = "https://github.com/FluidSynth/fluidsynth/raw/master/sf2/VintageDreamsWaves-v2.sf2"
        urllib.request.urlretrieve(url, soundfont_path)
    return soundfont_path

soundfont_path = download_soundfont()

# use example
abc_notation = dataset['train'][1]['output']
# Add L:1/8 to specify the default note length
abc_notation = "L:1/8\n" + abc_notation

try:
    wav_file = abc_to_wav(abc_notation)
    display(Audio(wav_file))
except FileNotFoundError:
    print("FluidSynth not found. Saving as MIDI file only.")
    # Parse the abc notation into a music21 stream
    abc_stream = converter.parse(abc_notation, format='abc')
    # Save the stream as a midi file
    midi_file = "output.mid"
    abc_stream.write('midi', fp=midi_file)
    # IPython.display.Audio does not accept a 'format' keyword; pass the filename only.
    display(Audio(midi_file))

FluidSynth not found. Saving as MIDI file only.


## Let's process the data for the learning task

Here I do a small clear of the inputs and the outputs i am going to use to train my model.

In [ ]:
def get_inputs_outputs(datas):
  inputs = [data["input"].replace("</s>","").strip() for data in datas]
  outputs = [data["output"].replace("Assistant:","").replace("</s>","").strip() for data in datas]
  return inputs, outputs

In [ ]:
# Let's take a smaller subset of the data for faster experimentation.
# The full dataset (~384k samples) requires a lot of memory and time.
n_samples_to_use = 10000 # You can adjust this number
datas = dataset['train'].select(range(n_samples_to_use))
inputs, outputs = get_inputs_outputs(datas)

In [ ]:
print(inputs[0],"\n")
outputs[0]

'A', 'D', 'E7', 'A', 'E/G#', 'A', 'Bm', 'A7/C#', 'D', 'E7', 'A', 'A', 'D', 'A', 'A', 'D', 'A', 'A', 'D', 'A', 'D', 'A/D#', 'E', 'A', 'D', 'A', 'A', 'D', 'A', 'E7' 



'X:1\nL:1/8\nM:2/2\nK:A\n CS ||"A" A,2 CE FECE | AECE FECE | A,2 CE FECE |"D" FAGF"E7" EDCB, |"A" A,2 CE FECE | AECE FEDC | \n"E/G#" B,D"A"CE"Bm" DF"A7/C#"E=G |"D" FA"E7"GB"A" cA A || f |"A" eA"D"fA"A" eAca | \n"A" eA"D"fA"A" e2 f/g/a |"A" eA"D"fA"A" eAce |"D" df"A/D#"ca"E" B2 d/e/f |"A" eA"D"fA"A" eAca | \n"A" eA"D"fA"A" e2 e/f/g | agag fedc |"E7" BAGF EDC!D.S.!B, |]'

As we can not train the model with character, we are going to map each into numbers.

In [ ]:
# To build the vocabulary without using excessive memory, we'll use a sample of the data.
datas_joined = "\n\n".join(inputs + outputs)

#Find all unique characters in the joined string
chars = sorted(list(set(datas_joined)))
chars.insert(0, ' ')
print("There are", len(chars), "unique characters in the dataset")

Building vocabulary from a sample of 10000 songs...
There are 97 unique characters in the dataset


### Vectorize the text
Before we begin training our model, we will need to create a numerical representation of our text-based dataset. To do this, we will generate two lookup tables: one table that maps characters to numbers and a second that maps numbers back to character.


In [ ]:
char2idx = {u:i for i,u in enumerate(chars)}
idx2char = np.array(chars)

In [ ]:
for char, _ in zip(char2idx, range(20)):
  print(f"{char}: {char2idx[char]}")



 : 2

: 1
!: 3
": 4
#: 5
$: 6
%: 7
&: 8
': 9
(: 10
): 11
*: 12
+: 13
,: 14
-: 15
.: 16
/: 17
0: 18
1: 19
2: 20


Given a input or an output (which are supposed to be string here), let's return its vectorized representation

In [ ]:
def vectorize_string(string):
  vectorized_output = np.array([char2idx[char] for char in string])
  return vectorized_output

vectorize_songs = vectorize_string(datas_joined)
print(datas_joined[:10],"-->",vectorize_songs[:10])

'A', 'D',  --> [ 9 34  9 14  2  9 37  9 14  2]


First we will train our model by using the logic of input-output like the classical way to predict adn after that we will explore another way to generate by using RNN

In [ ]:
# Get the maximum sequence lengths for padding
max_length_input = max(len(s) for s in inputs)
max_length_output = max(len(s) for s in outputs)
print(f"Max input length: {max_length_input}")
print(f"Max output length: {max_length_output}")

Max input length: 2885
Max output length: 3030


In [ ]:
# Pad vectorized sequences to fixed length so we can build a rectangular array
pad_idx = char2idx.get(" ", 0)  # use space index (or 0 if not present)

n_samples = len(inputs)

X_train = np.full((n_samples, max_length_input), fill_value=pad_idx, dtype=np.int64)
y_train = np.full((n_samples, max_length_output), fill_value=pad_idx, dtype=np.int64)

# Use a safe vectorizer that maps unknown characters to the pad index
def safe_vectorize_string(string, char2idx, pad_idx):
	return np.array([char2idx.get(ch, pad_idx) for ch in string], dtype=np.int64)

for i, inp in enumerate(inputs):
	vec = safe_vectorize_string(inp, char2idx, pad_idx)
	L = min(len(vec), max_length_input)
	if L > 0:
		X_train[i, :L] = vec[:L]

for i, out in enumerate(outputs):
	vec = safe_vectorize_string(out, char2idx, pad_idx)
	L = min(len(vec), max_length_output)
	if L > 0:
		y_train[i, :L] = vec[:L]

print("X_train shape:", X_train.shape, "y_train shape:", y_train.shape)

X_train shape: (383940, 2885) y_train shape: (383940, 3030)


#### Create training examples and targets
Our next step is to actually divide the text into example sequences that we'll use during training. Each input sequence that we feed into our RNN will contain seq_length characters from the text. We'll also need to define a target sequence for each input sequence, which will be used in training the RNN to predict the next character. For each input, the corresponding target will contain the same length of text, except shifted one character to the right.

In [ ]:
### Create PyTorch Dataset and DataLoader
# We'll use PyTorch's DataLoader to efficiently batch and shuffle the data during training.

from torch.utils.data import TensorDataset, DataLoader

# Convert numpy arrays to PyTorch tensors
X_train_tensor = torch.from_numpy(X_train).long()
y_train_tensor = torch.from_numpy(y_train).long()

# Create a dataset and dataloader
batch_size = 32 # Adjust based on your GPU memory
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Let's check a batch to see the shapes
data_iter = iter(train_loader)
inputs_batch, targets_batch = next(data_iter)
print("Input batch shape:", inputs_batch.shape)
print("Target batch shape:", targets_batch.shape)

Input batch shape: torch.Size([32, 2885])
Target batch shape: torch.Size([32, 3030])


### Define the Seq2Seq Model Architecture
A sequence-to-sequence (seq2seq) model consists of two main parts:
1.  **Encoder**: Reads the input sequence (chord progression) and compresses it into a fixed-size context vector (the 'thought' or 'meaning' of the sequence).
2.  **Decoder**: Takes the context vector from the encoder and generates the output sequence (the ABC notation), one character at a time.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, batch_first=True)

    def forward(self, src):
        # src shape: [batch_size, src_len]
        embedded = self.embedding(src) # shape: [batch_size, src_len, embedding_dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs are the top-layer hidden states from each time step
        # hidden and cell are the final hidden and cell states
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input, hidden, cell):
        # input shape: [batch_size] (current character)
        input = input.unsqueeze(1) # shape: [batch_size, 1]
        embedded = self.embedding(input) # shape: [batch_size, 1, embedding_dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output shape: [batch_size, 1, hidden_dim]
        prediction = self.fc_out(output.squeeze(1)) # shape: [batch_size, vocab_size]
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src shape: [batch_size, src_len]
        # trg shape: [batch_size, trg_len]
        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.vocab_size

        # tensor to store decoder outputs
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        # last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)

        # first input to the decoder is the start-of-sequence token from the target data
        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t] = output
            teacher_force = np.random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs

### Instantiate the Model and Define Training Components
Now we set the hyperparameters, create instances of our models, define the loss function, and choose an optimizer.

In [ ]:
# Model Hyperparameters
VOCAB_SIZE = len(chars)
EMBEDDING_DIM = 256
HIDDEN_DIM = 512
N_LAYERS = 2
LEARNING_RATE = 0.001

# Instantiate the model
encoder = Encoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS)
decoder = Decoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS)
model = Seq2Seq(encoder, decoder, device).to(device)

# Optimizer and Loss Function
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
# We use CrossEntropyLoss, but we need to ignore the padding index so it doesn't affect the loss.
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

### The Training Loop
This is where the magic happens. We'll iterate through our data for a few epochs, feeding it to the model, calculating the loss, and updating the model's weights.

**Note**: Training on the full dataset will take a very long time. The loop below is set to run for just one epoch and print the loss periodically. For a real training session, you would run for many more epochs.

In [ ]:
def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    
    for i, batch in enumerate(iterator):
        src = batch[0].to(device)
        trg = batch[1].to(device)
        
        optimizer.zero_grad()
        
        output = model(src, trg) # Teacher forcing is on by default
        
        # trg shape: [batch_size, trg_len]
        # output shape: [batch_size, trg_len, output_dim]
        
        output_dim = output.shape[-1]
        
        # Reshape for CrossEntropyLoss
        # We ignore the first token of the target sequence as it's our start token
        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        
        # Clip gradients to prevent them from exploding
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        
        optimizer.step()
        
        epoch_loss += loss.item()
        
        if (i+1) % 100 == 0:
            print(f'Batch {i+1}/{len(iterator)} | Batch Loss: {loss.item():.3f}')
        
    return epoch_loss / len(iterator)

# --- Start Training ---
N_EPOCHS = 1 # Increase for real training
CLIP = 1

print("Starting training...")
for epoch in range(N_EPOCHS):
    start_time = time.time()
    
    train_loss = train(model, train_loader, optimizer, criterion, CLIP)
    
    end_time = time.time()
    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)
    
    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs:.0f}s')
    print(f'\tTrain Loss: {train_loss:.3f}')

print("Training finished.")

Starting training...
Batch 100/11999 | Batch Loss: 2.581
Batch 200/11999 | Batch Loss: 2.389


KeyboardInterrupt: 

In [ ]:
# --- Save the model and necessary components ---

model_path = 'music_generator_model.pth'

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'char2idx': char2idx,
    'idx2char': idx2char,
    'max_length_input': max_length_input,
    'vocab_size': VOCAB_SIZE,
    'embedding_dim': EMBEDDING_DIM,
    'hidden_dim': HIDDEN_DIM,
    'n_layers': N_LAYERS
}, model_path)

print(f"Model saved to {model_path}")

### Load a Saved Model
Now, instead of retraining, you can simply load the saved file and use it directly for generation. This is what you would do in a new session.

In [ ]:
# --- Load the saved model ---
checkpoint = torch.load(model_path, map_location=device)

# Reconstruire l'architecture du modèle avec les hyperparamètres sauvegardés
loaded_encoder = Encoder(checkpoint['vocab_size'], checkpoint['embedding_dim'], checkpoint['hidden_dim'], checkpoint['n_layers'])
loaded_decoder = Decoder(checkpoint['vocab_size'], checkpoint['embedding_dim'], checkpoint['hidden_dim'], checkpoint['n_layers'])
loaded_model = Seq2Seq(loaded_encoder, loaded_decoder, device).to(device)

# Charger les poids entraînés
loaded_model.load_state_dict(checkpoint['model_state_dict'])

print("Model loaded successfully from disk.")

### Generate New Music from a Chord Progression

Now for the fun part! We can use our trained model to generate a new musical piece from a sequence of chords. The function below orchestrates the inference process:
1. It takes a string of chords as input.
2. It vectorizes and pads the input, just like we did for the training data.
3. It feeds the input through the `encoder` to get the context vectors.
4. It uses the `decoder` in a loop, starting with an 'X' token, to generate the output sequence character by character.
5. It converts the generated indices back to a string and returns the result.

In [ ]:
def generate_music(model, input_chords, char2idx, idx2char, device, max_input_len, max_output_len=500):
    model.eval() # Set the model to evaluation mode

    # Vectorize and pad the input chord string
    pad_idx = char2idx.get(" ", 0)
    tokenized_input = vectorize_string(input_chords)
    padded_input = np.full((1, max_input_len), fill_value=pad_idx, dtype=np.int64)
    L = min(len(tokenized_input), max_input_len)
    padded_input[0, :L] = tokenized_input[:L]
    src_tensor = torch.from_numpy(padded_input).long().to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    # Start decoding with the 'X' character (common start of ABC notation)
    trg_indexes = [char2idx['X']]

    for i in range(max_output_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)

        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
        
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        # We can define a stop condition, e.g., the end of a tune in ABC is `|]`
        # For simplicity, we'll just generate up to max_output_len here.
        # A more robust solution would check for an end-of-sequence token.

    trg_tokens = [idx2char[i] for i in trg_indexes]
    return "".join(trg_tokens)

# --- Example Usage ---
# Define a new chord progression
new_chords = "'G', 'C', 'G', 'D7', 'G', 'C', 'G', 'D7', 'G'"

print(f"Generating music for chords: {new_chords}\n")

# Generate the ABC notation
generated_abc = generate_music(loaded_model, new_chords, checkpoint['char2idx'], checkpoint['idx2char'], device, max_input_len=checkpoint['max_length_input'])

print("--- Generated ABC Notation ---")
print(generated_abc)